# Transition experiment: does temporal context help label segments near seizure boundaries?

In the earlier ablation, every window was entirely seizure or entirely normal, and the TCN never helped.
Here each window is cut into **1 s segments with their own label** (normal / seizure), and windows **cross seizure onsets and offsets**.
The model (`AsymSETNetSegments`) outputs one prediction per segment:

| Variant | Temporal context |
|---|---|
| `0` | none: each segment is classified from its own spatial features |
| `Lc` (L = 1, 2, 3) | causal TCN: a segment sees only past segments (like a real-time detector) |
| `Lnc` | non-causal TCN: a segment sees past and future segments (offline labelling) |

**Protocol** (`datasets/transition.py → fold_windows`):
- **Data:** the patient's seizure recordings as a continuous signal, with a label per second.
- **Folds:** 5 folds by recording. Patients with fewer than 5 recordings use leave-one-recording-out.
- **Train and validation:** windows with a stride of 1 s.
  - Validation is 20 % of the ~60 s time blocks in the training recordings, stratified by seizure. There's no overlap between the two sets.
  - All-normal windows are capped at 5 × the number of windows that contain seizure, per recording.
- **Test:** each test recording is tiled with non-overlapping windows, so every second is scored once. The same cap applies, but tiles touching a boundary are always kept.
- **Training:** Focal Loss on every segment. Predictions use a threshold of 0.5.

**Metrics:** per-segment TP / FP / TN / FN and F1 on **all** test segments, and on **boundary** segments, i.e. the seconds within ±`BOUNDARY_K` s of an onset or offset. The boundary table answers the question.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "experiments" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))            # metapaper/ first (wins over the pip `datasets` package)

from datasets import transition as tr
from models.asymsetnet import AsymSETNetSegments
from experiments.training import (DEVICE, TrainConfig, set_seed, train_model,
                                  predict_proba, confusion_counts, add_metrics)

warnings.filterwarnings("ignore", message="Only .* recordings")   # LORO fallback, expected
print("root:", ROOT, "| device:", DEVICE)

## Configuration

In [ ]:
PATIENTS = ["chb01"]                 # all: tr.available_patients()
WINDOW_SECONDS = [3, 5, 10]          # 1 s segments -> N = window_sec segments
TEMPORAL_LAYERS = [0, 1, 2, 3]
CAUSAL = [True, False]               # 0 layers has no causal variant

N_SPLITS = 5
STRIDE = 1                           # train/val window stride (s)
BLOCK_SEC = 60                       # validation time-block length (s)
VAL_FRAC = 0.2
MAX_RATIO = 5                        # all-normal windows <= 5 x windows with seizure
BOUNDARY_K = 2                       # boundary segments: within +-2 s of an onset/offset
TCN_WIDTH = 32
THRESHOLD = 0.5
SEED = 0

CFG = TrainConfig(epochs=100, patience=15, batch_size=64, lr=1e-3, weight_decay=1e-4,
                  focal_alpha=0.75, focal_gamma=2.0)

# model variants: (name, n_layers, causal)
VARIANTS = [("0", 0, True)] + [(f"{L}{'c' if c else 'nc'}", L, c)
                               for L in TEMPORAL_LAYERS if L > 0 for c in CAUSAL]

RESULTS_DIR = ROOT / "experiments" / "results"
TAG = "transition_segments"
PARTIAL_CSV = RESULTS_DIR / f"{TAG}_folds_partial.csv"   # progress checkpoint
print([v[0] for v in VARIANTS])

## Run

After each (patient, window, fold), results are saved to `PARTIAL_CSV`. Rerunning the cell skips what's already done. Delete that file to start from scratch.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
rows = pd.read_csv(PARTIAL_CSV).to_dict("records") if PARTIAL_CSV.exists() else []
done = {(r["patient"], r["window_sec"], r["variant"], r["fold"]) for r in rows}
print(f"resuming with {len(rows)} results" if rows else "starting fresh")

t_start = time.time()
for patient in PATIENTS:
    d = tr.load_patient(patient)
    folds = list(tr.split_folds(d, N_SPLITS, seed=SEED))
    print(f"{patient}: {len(d['rec_names'])} recordings, {len(d['sec_label']) / 3600:.1f} h, "
          f"{int(d['sec_label'].sum())} seizure s, {len(folds)} folds")

    for window_sec in WINDOW_SECONDS:
        t0 = time.time()
        for fold, (train_recs, test_recs) in enumerate(folds):
            todo = [v for v in VARIANTS if (patient, window_sec, v[0], fold) not in done]
            if not todo:
                continue
            w = tr.fold_windows(d, train_recs, test_recs, window_sec, stride=STRIDE,
                                max_ratio=MAX_RATIO, block_sec=BLOCK_SEC, val_frac=VAL_FRAC,
                                boundary_k=BOUNDARY_K, seed=SEED + fold)
            Y_te, B_te = w["test"]["Y"], w["test"]["boundary"]

            for name, n_layers, causal in todo:
                set_seed(SEED + fold)
                model = AsymSETNetSegments(n_channels=w["train"]["X"].shape[1], n_segments=window_sec,
                                           tcn_channels=(TCN_WIDTH,) * n_layers, causal=causal)
                model, best_epoch, _ = train_model(model, w["train"]["X"], w["train"]["Y"],
                                                   w["val"]["X"], w["val"]["Y"], CFG)
                pred = (predict_proba(model, w["test"]["X"]) >= THRESHOLD).astype(int)

                all_c = confusion_counts(Y_te.ravel(), pred.ravel())
                bnd_c = confusion_counts(Y_te[B_te], pred[B_te])
                rows.append({
                    "patient": patient, "window_sec": window_sec, "variant": name,
                    "temporal_layers": n_layers, "causal": causal if n_layers else None, "fold": fold,
                    "test_recordings": ", ".join(str(d["rec_names"][r]) for r in test_recs),
                    "n_train_win": len(w["train"]["Y"]), "n_val_win": len(w["val"]["Y"]),
                    "n_test_seg": Y_te.size, "n_boundary_seg": int(B_te.sum()),
                    **all_c, **{f"b{k}": v for k, v in bnd_c.items()},
                    "best_epoch": best_epoch,
                })
            pd.DataFrame(rows).to_csv(PARTIAL_CSV, index=False)   # checkpoint

        res = pd.DataFrame([r for r in rows if r["patient"] == patient and r["window_sec"] == window_sec])
        summ = res.groupby("variant")[["TP", "FP", "TN", "FN", "bTP", "bFP", "bTN", "bFN"]].sum()
        f1 = lambda tp, fp, fn: 2 * tp / max(2 * tp + fp + fn, 1)
        print(f"  {window_sec:2d}s ({time.time() - t0:.0f}s)  F1 all / boundary: " + "  ".join(
            f"{v}={f1(s.TP, s.FP, s.FN):.2f}/{f1(s.bTP, s.bFP, s.bFN):.2f}"
            for v, s in summ.loc[[v[0] for v in VARIANTS if v[0] in summ.index]].iterrows()))

results = pd.DataFrame(rows)
print(f"done: {len(results)} results, {(time.time() - t_start) / 60:.1f} min")

## Results

`add_metrics` works on the TP/FP/TN/FN columns. `boundary_metrics` computes the same metrics from the boundary counts (`bTP`, …).

In [ ]:
VARIANT_ORDER = [v[0] for v in VARIANTS]
COUNTS = ["TP", "FP", "TN", "FN"]
B_COUNTS = ["bTP", "bFP", "bTN", "bFN"]

def boundary_metrics(df):
    b = add_metrics(df[B_COUNTS].set_axis(COUNTS, axis=1))
    return b.drop(columns=COUNTS).add_prefix("boundary_")

def with_metrics(df):
    return pd.concat([add_metrics(df), boundary_metrics(df)], axis=1)

per_fold = with_metrics(results)
keys = ["patient", "window_sec", "variant"]
per_config = with_metrics(results.groupby(keys)[COUNTS + B_COUNTS].sum().reset_index())
per_config.style.format(precision=3).hide(axis="index")

### Final tables: F1 per variant × window length

**Across patients:**
- **Mean F1:** the mean over patients of each patient's pooled F1, so every patient weighs the same.
- **Pooled F1:** computed from counts summed over every patient and fold, so patients with more seizures weigh more. For the boundary table this is the most reliable number: a single patient has very few boundary segments (e.g. 16 per test fold for chb01).

In [ ]:
def grid(df, col, agg="mean"):
    t = df.pivot_table(index="variant", columns="window_sec", values=col, aggfunc=agg)
    return t.reindex([v for v in VARIANT_ORDER if v in t.index])

pooled_all = with_metrics(results.groupby(["window_sec", "variant"])[COUNTS + B_COUNTS].sum().reset_index())

tables = {
    "F1 all segments - mean across patients": grid(per_config, "f1"),
    "F1 boundary segments - mean across patients": grid(per_config, "boundary_f1"),
    "F1 all segments - pooled": grid(pooled_all, "f1"),
    "F1 boundary segments - pooled": grid(pooled_all, "boundary_f1"),
    "Sensitivity boundary segments - pooled": grid(pooled_all, "boundary_sensitivity"),
    "Specificity boundary segments - pooled": grid(pooled_all, "boundary_specificity"),
}
print("patients:", results["patient"].nunique())
for name, t in tables.items():
    print(name)
    display(t.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold"))

Per patient, boundary F1:

In [ ]:
for patient, df in per_config.groupby("patient"):
    print(patient)
    display(grid(df, "boundary_f1").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold"))

Save everything:

In [ ]:
per_fold.to_csv(RESULTS_DIR / f"{TAG}_folds.csv", index=False)
per_config.to_csv(RESULTS_DIR / f"{TAG}_configs.csv", index=False)
for name, t in tables.items():
    slug = name.lower().replace(" - ", "_").replace(" ", "_")
    t.to_csv(RESULTS_DIR / f"{TAG}_{slug}.csv")
print("saved ->", RESULTS_DIR / f"{TAG}_*.csv")